# Basic Transformation: Quantum Fourier Transform — Qiskit Notebook

**Chapter / concept:** Basic transformations — the **standard 3-qubit Quantum Fourier
Transform (QFT)**, including the conventional trailing SWAP network.

**Transformation implemented:**
$$\mathrm{QFT}|x\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} e^{2\pi i\, x k / N}\,|k\rangle,
\qquad N = 2^3 = 8$$

applied to the concrete computational-basis input **x = 5** (bitstring `101`), so the
notebook shows genuinely non-trivial phases rather than the flat, phase-free case x = 0.

### Global SETUP (identical in both notebooks)
- **N_QUBITS** = 3
- **Input state:** x = 5 → prepare `|101>` via `X` on q0 and q2 (q1 stays `|0>`)
- **Gate sequence (QFT proper, after input prep):**
  `H(q2) → CP(2π/4; q1→q2) → CP(2π/8; q0→q2) → H(q1) → CP(2π/4; q0→q1) → H(q0) → SWAP(q0,q2)`
- **QUBIT ORDERING CONTRACT:** qubit 0 = rightmost bit in every printed bitstring. This is
  Qiskit's native convention — no reversal needed here. (The CUDA-Q notebook reverses its
  bitstrings to match.)
- **SHOTS** = 10000, **SEED** = 42
- **GLOBAL PHASE CONVENTION:** normalize so the first amplitude with `abs(amp) > 1e-9`, in
  ascending bitstring order, is rotated to be real and positive. For any QFT output this is
  automatically a no-op: the k=0 amplitude of `QFT|x>` is always `1/sqrt(N)` (real, positive)
  for every input x, by the formula above.
- **Basis ordering for matrices:** `{|000>, |001>, ..., |111>}` ascending binary order.

### Theory recap
The standard QFT circuit processes qubits from the most-significant (q2) down to the
least-significant (q0): put `H` on a qubit, then apply a controlled-phase gate from every
*lower*-index qubit still to be processed, with angle `2π/2^k` where `k` is the "distance"
between the two qubits (k=2 for adjacent qubits, k=3 for two apart, etc). After processing
all qubits this way, a final SWAP network reverses the qubit order so the output basis index
`k` lines up with the same qubit-ordering convention as the input `x`. This exact circuit
(gate-by-gate, including the SWAP) was independently re-derived and numerically verified
against the closed-form DFT matrix above — for the full 8x8 unitary AND for the specific
`x=5` output amplitudes — using a hand-written NumPy circuit simulator (same
bit-ordering/phase convention as this notebook), to machine precision, before being used here.

**Run all cells top to bottom.**


In [ ]:
# Install cell
# Package names: `qiskit` (core) and `qiskit-aer` (the Aer simulator backend).
# Tested against the current stable Qiskit 1.x line; qiskit-aer provides AerSimulator.
#
# IMPORTANT: plain `!pip install ...` can silently install into a DIFFERENT Python
# environment than the one this notebook's kernel is actually running (common on
# Colab/Jupyter setups with more than one interpreter on PATH) -- that mismatch is what
# causes "ModuleNotFoundError: No module named 'qiskit'" on the very next cell even
# though this install cell appears to run without errors. Using
# `{sys.executable} -m pip` pins the install to the exact interpreter this kernel uses.
import sys
!{sys.executable} -m pip install -q qiskit qiskit-aer pylatexenc


In [ ]:
# Imports + constants
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

N_QUBITS = 3
X_INPUT = 5                 # the computational-basis input |x=5> = |101>
SHOTS = 10000
SEED = 42
TOPIC_SLUG = "qft"

# Controlled-phase angles used by the standard QFT circuit: angle = 2*pi / 2^k
THETA_K2 = 2 * np.pi / (2**2)   # pi/2  -- used between adjacent qubits
THETA_K3 = 2 * np.pi / (2**3)   # pi/4  -- used between qubits 2 apart

print(f"N_QUBITS = {N_QUBITS}")
print(f"X_INPUT  = {X_INPUT}  (bitstring {format(X_INPUT, '03b')})")
print(f"SHOTS    = {SHOTS}")
print(f"SEED     = {SEED}")
print(f"THETA_K2 = {THETA_K2:.6f} rad (pi/2)")
print(f"THETA_K3 = {THETA_K3:.6f} rad (pi/4)")


In [ ]:
# Build the circuit gate-by-gate, with a physical comment on each gate.
# Split into an input-prep sub-circuit and a QFT-proper sub-circuit so the QFT transform
# itself can be matrix-checked in isolation later.

state_prep = QuantumCircuit(N_QUBITS, name="state_prep")
# Prepare |x=5> = |101> (q2=1, q1=0, q0=1): flip q0 and q2 from |0> to |1>.
state_prep.x(0)
state_prep.x(2)

qft_only = QuantumCircuit(N_QUBITS, name="QFT")
# --- process qubit 2 (most significant) ---
# H on q2: starts mixing q2's phase information into an equal superposition.
qft_only.h(2)
# Controlled phase from q1 (distance 1 -> k=2, angle 2pi/4): kicks a phase onto q2's |1>
# component conditioned on q1, encoding q1's contribution to q2's Fourier phase.
qft_only.cp(THETA_K2, 1, 2)
# Controlled phase from q0 (distance 2 -> k=3, angle 2pi/8): same idea, smaller phase kick
# since q0 is "farther" from q2 in significance.
qft_only.cp(THETA_K3, 0, 2)
# --- process qubit 1 ---
qft_only.h(1)
# Controlled phase from q0 (distance 1 -> k=2, angle 2pi/4).
qft_only.cp(THETA_K2, 0, 1)
# --- process qubit 0 (least significant) ---
qft_only.h(0)
# --- final SWAP network: reverses qubit order so output index k uses the same
#     qubit-ordering convention as input x. For n=3 there's a single swap: (q0, q2).
qft_only.swap(0, 2)

qc = state_prep.compose(qft_only)
print(qc.draw(output="text"))


In [ ]:
# Draw the circuit with matplotlib and save to file
fig = qc.draw(output="mpl")
fig.savefig(f"{TOPIC_SLUG}_qiskit_circuit.png", dpi=150, bbox_inches="tight")
fig


In [ ]:
# Explicit matrix representation: constituent gate matrices + combined QFT operator,
# built two ways and cross-checked (checking the QFT-proper sub-circuit in isolation,
# i.e. NOT including the input-prep X gates, against the closed-form DFT matrix).

H_MAT = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]])
I_MAT = np.eye(2)

def phase_gate(theta):
    return np.array([[1, 0], [0, np.exp(1j * theta)]])

print("H matrix:")
print(np.round(H_MAT, 4))
print(f"\nPhase gate P(pi/2) matrix:")
print(np.round(phase_gate(THETA_K2), 4))
print(f"\nPhase gate P(pi/4) matrix:")
print(np.round(phase_gate(THETA_K3), 4))

def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out

def single(gate_2x2, qubit, n):
    mats = [gate_2x2 if pos == qubit else I_MAT for pos in range(n - 1, -1, -1)]
    return kron_all(mats)

def controlled_phase(theta, control, target, n):
    P1 = np.array([[0, 0], [0, 1]])
    P0 = np.array([[1, 0], [0, 0]])
    Pg = phase_gate(theta)
    mats_match, mats_other = [], []
    for pos in range(n - 1, -1, -1):
        if pos == control:
            mats_match.append(P1); mats_other.append(P0)
        elif pos == target:
            mats_match.append(Pg); mats_other.append(I_MAT)
        else:
            mats_match.append(I_MAT); mats_other.append(I_MAT)
    return kron_all(mats_match) + kron_all(mats_other)

def swap_op(q1, q2, n):
    dim = 2**n
    op = np.zeros((dim, dim))
    for idx in range(dim):
        bits = [(idx >> b) & 1 for b in range(n)]
        bits[q1], bits[q2] = bits[q2], bits[q1]
        new_idx = sum(bits[b] << b for b in range(n))
        op[new_idx, idx] = 1.0
    return op

# (a) Hand-built unitary, applied in the same order as the circuit above.
hand_built_qft = single(H_MAT, 2, N_QUBITS)
hand_built_qft = controlled_phase(THETA_K2, 1, 2, N_QUBITS) @ hand_built_qft
hand_built_qft = controlled_phase(THETA_K3, 0, 2, N_QUBITS) @ hand_built_qft
hand_built_qft = single(H_MAT, 1, N_QUBITS) @ hand_built_qft
hand_built_qft = controlled_phase(THETA_K2, 0, 1, N_QUBITS) @ hand_built_qft
hand_built_qft = single(H_MAT, 0, N_QUBITS) @ hand_built_qft
hand_built_qft = swap_op(0, 2, N_QUBITS) @ hand_built_qft

# (b) Qiskit's own operator extraction, on the QFT-proper sub-circuit only.
qiskit_qft_operator = Operator(qft_only).data

print("\nHand-built QFT unitary, rounded (first 4x4 corner shown for brevity):")
print(np.round(hand_built_qft[:4, :4], 4))
print("\nQiskit Operator(qft_only), rounded (first 4x4 corner shown for brevity):")
print(np.round(qiskit_qft_operator[:4, :4], 4))

max_op_diff = np.max(np.abs(hand_built_qft - qiskit_qft_operator))
print(f"\nMax elementwise difference between hand-built and Qiskit operator (full 8x8): {max_op_diff:.2e}")
assert max_op_diff < 1e-8, "Hand-built and Qiskit QFT operators disagree!"
print("Cross-check PASSED: hand-built matrix matches Qiskit's Operator extraction.")

# (c) Also cross-check against the closed-form DFT matrix definition directly.
N = 2**N_QUBITS
dft_ref = np.zeros((N, N), dtype=complex)
for k in range(N):
    for x in range(N):
        dft_ref[k, x] = np.exp(2j * np.pi * x * k / N) / np.sqrt(N)
max_dft_diff = np.max(np.abs(hand_built_qft - dft_ref))
print(f"Max difference vs closed-form DFT matrix: {max_dft_diff:.2e}")
assert max_dft_diff < 1e-8, "Circuit does not implement the standard DFT matrix!"
print("Cross-check PASSED: circuit implements the standard QFT/DFT matrix exactly.")


In [ ]:
# Exact statevector: extract, apply global-phase normalization, print amplitudes + probs

def normalize_global_phase(statevector, atol=1e-9):
    sv = np.array(statevector, dtype=complex).copy()
    for amp in sv:
        if abs(amp) > atol:
            phase = np.angle(amp)
            sv = sv * np.exp(-1j * phase)
            break
    return sv

raw_sv = Statevector(qc).data
sv = normalize_global_phase(raw_sv)   # Qiskit: no bit-order reversal needed here

n = N_QUBITS
qiskit_prob_dict = {}
print("Statevector amplitudes (bitstring = q(n-1)...q0, qubit0 rightmost):")
for idx in range(2**n):
    bitstring = format(idx, f"0{n}b")
    amp = sv[idx]
    prob = abs(amp) ** 2
    qiskit_prob_dict[bitstring] = prob
    print(f"  |{bitstring}> (k={idx})  amp = {amp:.4f}   prob = {prob:.4f}")

qiskit_prob_dict = dict(sorted(qiskit_prob_dict.items()))
print("\nAs expected for any QFT of a computational basis state, all 8 probabilities are "
      "equal (1/8) -- only the phases encode which input x was transformed:")
print(f"  min prob = {min(qiskit_prob_dict.values()):.6f}, max prob = {max(qiskit_prob_dict.values()):.6f}")


### Ordering-fix note (Qiskit)

**No bit reversal is needed in this notebook.** Qiskit already indexes both statevector
amplitudes and measurement counts with **qubit 0 as the least-significant (rightmost) bit**,
matching the qubit-ordering contract this exercise standardizes on. The CUDA-Q notebook's
qubit 0 is the opposite (most-significant/leftmost) by default, so that notebook performs an
explicit reversal to match this one.

**Phase convention applied:** the statevector above was rotated so the first amplitude with
`abs(amp) > 1e-9`, in ascending bitstring order, is real and positive. This is automatically
a no-op for a QFT output: `amp(k=0) = 1/sqrt(N)` for every input `x`, which is already real
and positive.


In [ ]:
# Measurement cell: sample with the shared SEED/SHOTS, get counts, convert to probabilities
meas_qc = qc.copy()
meas_qc.measure_all()

sim = AerSimulator(seed_simulator=SEED)
# Transpile first so any high-level instruction (e.g. `cp`, `swap`) is decomposed into
# gates Aer's internal compiler natively supports before running -- avoids
# "AerError: unknown instruction: ..." style failures.
transpiled_meas_qc = transpile(meas_qc, sim)
result = sim.run(transpiled_meas_qc, shots=SHOTS, seed_simulator=SEED).result()
counts = result.get_counts()

qiskit_sampled_probs = {bitstring: count / SHOTS for bitstring, count in counts.items()}
qiskit_sampled_probs = dict(sorted(qiskit_sampled_probs.items()))

print("Raw counts:", dict(sorted(counts.items())))
print("\nSampled probabilities (sorted by bitstring) -- measurement destroys the phase "
      "information, so this should come out close to a flat 1/8 for every outcome:")
print({k: round(v, 4) for k, v in qiskit_sampled_probs.items()})


In [ ]:
# Histogram: matplotlib bar chart, saved to file
fig2, ax2 = plt.subplots(figsize=(8, 4))
all_bitstrings = [format(i, f"0{N_QUBITS}b") for i in range(2**N_QUBITS)]
values = [qiskit_sampled_probs.get(b, 0.0) for b in all_bitstrings]
ax2.bar(all_bitstrings, values, color="#6FA8DC", edgecolor="black")
ax2.set_ylim(0, 0.3)
ax2.set_xlabel("bitstring (q2 q1 q0), i.e. output index k")
ax2.set_ylabel("probability")
ax2.set_title("QFT|x=5> sampled probabilities — Qiskit")
plt.xticks(rotation=45)
fig2.tight_layout()
fig2.savefig(f"{TOPIC_SLUG}_qiskit_histogram.png", dpi=150, bbox_inches="tight")
fig2


In [ ]:
# Resource counts
depth = qc.depth()
gate_counts = qc.count_ops()
print(f"Circuit depth: {depth}")
print(f"Gate counts: {dict(gate_counts)}")


### Convention notes (restated)

- **Ordering fix applied:** none — Qiskit is the reference convention (qubit 0 = rightmost
  bit) that the CUDA-Q notebook reverses its own output to match.
- **Phase convention applied:** global phase rotated so the first nonzero amplitude in
  ascending bitstring order is real and positive (a no-op here, as explained above).
- **Bonus cross-check (QFT only):** the hand-built matrix for the QFT-proper sub-circuit
  was checked against both Qiskit's `Operator` extraction AND the closed-form DFT matrix
  definition directly, and the resulting `x=5` amplitudes were checked against the analytic
  formula `exp(2*pi*i*x*k/N)/sqrt(N)` (see the diff-check cell below).


In [ ]:
# Final result dictionary
QISKIT_RESULT = qiskit_sampled_probs
print("QISKIT_RESULT =")
print({k: round(v, 4) for k, v in QISKIT_RESULT.items()})


In [ ]:
# Diff-check block: hand-computed expected amplitudes/probabilities from the closed-form
# QFT formula, plus (commented) live comparison against the CUDA-Q notebook's result.

N = 2**N_QUBITS
HAND_COMPUTED_AMPLITUDES = {}
HAND_COMPUTED_EXPECTED = {}
for k in range(N):
    bitstring = format(k, f"0{N_QUBITS}b")
    amp = np.exp(2j * np.pi * X_INPUT * k / N) / np.sqrt(N)
    HAND_COMPUTED_AMPLITUDES[bitstring] = amp
    HAND_COMPUTED_EXPECTED[bitstring] = abs(amp) ** 2   # == 1/N for every k

print(f"Hand-computed expected amplitudes for QFT|x={X_INPUT}>:")
for k_str, amp in HAND_COMPUTED_AMPLITUDES.items():
    print(f"  |{k_str}>  amp={amp:.4f}")

print("\nHand-computed expected (ideal, noiseless) probabilities -- all equal to 1/8:")
print({k: round(v, 4) for k, v in HAND_COMPUTED_EXPECTED.items()})

max_delta_vs_ideal = max(abs(QISKIT_RESULT.get(k, 0.0) - v) for k, v in HAND_COMPUTED_EXPECTED.items())
print(f"\nMax |sampled - ideal| delta (Qiskit, {SHOTS} shots): {max_delta_vs_ideal:.4f}")
# With finite shots this will be within normal statistical noise, not exactly 0.

# Cross-check the actual statevector amplitudes (not just probabilities) against the formula:
amp_diff = np.max(np.abs(sv - np.array([HAND_COMPUTED_AMPLITUDES[format(k, f'0{N_QUBITS}b')] for k in range(N)])))
print(f"Max |statevector - formula| amplitude difference: {amp_diff:.2e}")
assert amp_diff < 1e-8, "Statevector amplitudes do not match the closed-form QFT formula!"
print("Cross-check PASSED: statevector amplitudes match the analytic QFT formula exactly.")

# CUDAQ_RESULT = { ... }  # paste from the CUDA-Q notebook
# max_delta = max(abs(QISKIT_RESULT[k] - CUDAQ_RESULT[k]) for k in QISKIT_RESULT)
# print(f"Max delta QISKIT_RESULT vs CUDAQ_RESULT: {max_delta:.4f}")
# assert max_delta < 1e-1, "Qiskit and CUDA-Q results diverge beyond shot noise!"
